# Полный пайплайн

In [ ]:
'''
Что уже известно (ноутбуки _data_overview, _boosting_baseline, _shifted_slices):
- данные чистые, 100 000 клиентов по одному срезу, история обрезана по дате отсечения, утечек в ID нет;
- 222 признака + LightGBM и CatBoost: 0.722 на кросс-валидации, 0.718 на Public; шум Public ± 0.017;
- главный сигнал — ритм заказов; у клиентов с 2–3 заказами target почти предопределён днём
  «последний заказ + средний интервал»; для 4+ заказов такого правила нет;
- важна вся двухлетняя история: обрезка до 532 дней стоит около 0.05;
- метки, посчитанные по истории на сдвинутых датах, не помогают ни в каком виде;
- признаки на плато: 452 признака дают то же, что 222; вторая модель внутри верхних 30% рейтинга
  не улучшает первую; двойников среди клиентов нет; зависимость устойчива между срезами;
- часть признаков зависит от длины целевого окна (90, 89, 92 дня у срезов train, 91 у теста)
  и у очень частых покупателей фактически кодирует номер среза.

Что делает этот ноутбук:
1. признаки на полной истории; проверка варианта с одной фиксированной длиной окна для всех срезов;
2. схема проверки: кросс-валидация с повторами на январе и феврале, март — нетронутый контроль,
   число деревьев определяется по внутренней отложенной части, без подглядывания в проверочный фолд;
3. отбор признаков сравнением с важностью на перемешанном таргете;
4. LightGBM, XGBoost, CatBoost и LightGBM с отдельными моделями для 1–3 и 4+ дней с заказами;
5. подбор параметров Optuna на январе и феврале; лучшие попытки перепроверяются с повторами и на марте,
   и принимаются только если они не хуже исходных параметров на обоих;
6. смесь моделей: веса подбираются на январе и феврале и проверяются на марте;
7. финал: обучение на всём train с усреднением по зёрнам, две отправки, журнал экспериментов.

Запуск: Save & Run All, ускоритель T4. QUICK = True — пробный прогон на несколько минут.
'''
import itertools
import json
import os
import re
import subprocess
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import xgboost as xgb
from catboost import CatBoostClassifier
from IPython.display import display
from scipy.special import ndtr
from sklearn.model_selection import StratifiedKFold, train_test_split

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

KAGGLE_DIR = Path("/kaggle/input/competitions/lamoda-cu-repeat-purchase-2026")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else Path(os.environ.get("LAMODA_DATA_DIR", "data"))
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("LAMODA_OUT_DIR", "."))

QUICK = os.environ.get("LAMODA_QUICK", "0") == "1"   # поставь True для пробного прогона
RUN_WINDOW_CHECK = True
WIN_FIXED = 91                         # длина окна теста; проверяется как общая для всех срезов
RUN_SELECTION = True
RUN_TUNING = True
SEED = 42
GAP_DAYS = 14
TARGET_MONTHS = 3
WINDOWS = [7, 14, 30, 60, 90, 180, 365]
N_FOLDS = 5
N_REPEATS = 1 if QUICK else 2          # повторы кросс-валидации при перепроверке
TUNE_FOLDS = 3                         # фолды внутри подбора
N_TRIALS = {"lgb": 3, "xgb": 3, "cb": 2} if QUICK else {"lgb": 40, "xgb": 40, "cb": 30}
TUNE_TIMEOUT = 120 if QUICK else 45 * 60   # предел времени подбора на одну модель, секунд
N_TOP = 1 if QUICK else 3              # сколько лучших попыток перепроверять
N_HOLD_SEEDS = 1 if QUICK else 2       # зёрна при проверке на марте
N_SEEDS = 2 if QUICK else 5            # зёрна в финальном обучении
EARLY_STOP = 150
ACCEPT_TOL = 0.0005                    # допустимое ухудшение на контроле при принятии изменения


def has_gpu():
    try:
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except Exception:
        return False


GPU = has_gpu()
print("GPU:", GPU, "| QUICK:", QUICK)
T0 = time.time()
LOG = []


def log(stage, name, **kw):
    LOG.append({"минута": round((time.time() - T0) / 60, 1), "стадия": stage, "что": name, **kw})


def fast_ap(target, score):
    order = np.argsort(-np.asarray(score), kind="mergesort")
    t = np.asarray(target)[order]
    k = int(np.ceil(len(t) * 0.20))
    top = t[:k]
    return float((np.cumsum(top) / np.arange(1, k + 1) * top).sum() / max(min(k, int(t.sum())), 1))

## Данные и признаки

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["scoring_date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["scoring_date"])
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
orders = pd.read_parquet(DATA_DIR / "orders.parquet")
products = pd.read_parquet(DATA_DIR / "products.parquet")
orders["order_date"] = pd.to_datetime(orders["order_date"])

cust = pd.concat([train.assign(part="train"), test.assign(part="test", target=np.nan)], ignore_index=True)
cust["slice"] = cust["scoring_date"].dt.strftime("%Y-%m-%d")
cust["cutoff"] = cust["scoring_date"] - pd.Timedelta(days=GAP_DAYS)
cust["win_len"] = cust["scoring_date"].map(lambda d: (d + pd.DateOffset(months=TARGET_MONTHS) - d).days)
assert cust["customer_id"].is_unique

lines = orders.merge(products, on="product_id", how="left", validate="m:1").merge(
    cust[["customer_id", "cutoff"]], on="customer_id", how="inner", validate="m:1"
)
lines["d"] = (lines["cutoff"] - lines["order_date"]).dt.days
assert len(lines) == len(orders) and lines["d"].between(0, 732).all()


def clean(name):
    return re.sub(r"\W+", "_", str(name)).strip("_").lower()


def build_features(lines, win_len):
    lines = lines.copy()
    lines["has_disc"] = (lines["discount_percent"] > 0).astype("float32")
    lines["has_loy"] = (lines["loyalty_percent"] > 0).astype("float32")
    lines["weekend"] = (lines["order_date"].dt.dayofweek >= 5).astype("float32")

    # --- уровень заказа, самые свежие заказы идут первыми
    ot = (
        lines.groupby(["customer_id", "order_id"], sort=False)
        .agg(
            d=("d", "first"),
            order_sum=("paid_price", "sum"),
            n_items=("paid_price", "size"),
            disc=("discount_percent", "mean"),
            loy=("loyalty_percent", "max"),
            weekend=("weekend", "first"),
        )
        .reset_index()
        .sort_values(["customer_id", "d", "order_id"])
        .reset_index(drop=True)
    )
    cid = ot["customer_id"]
    f = ot.groupby("customer_id").agg(
        recency=("d", "min"),
        tenure=("d", "max"),
        d_mean=("d", "mean"),
        d_std=("d", "std"),
        n_orders=("order_id", "size"),
        n_items=("n_items", "sum"),
        sum_paid=("order_sum", "sum"),
        order_sum_mean=("order_sum", "mean"),
        order_sum_median=("order_sum", "median"),
        order_sum_max=("order_sum", "max"),
        order_sum_std=("order_sum", "std"),
        items_per_order=("n_items", "mean"),
        items_max=("n_items", "max"),
        order_disc_mean=("disc", "mean"),
        order_loy_mean=("loy", "mean"),
        order_loy_max=("loy", "max"),
        weekend_share=("weekend", "mean"),
    )

    last = ot.drop_duplicates("customer_id").set_index("customer_id")
    first = ot.drop_duplicates("customer_id", keep="last").set_index("customer_id")
    f["last_order_sum"] = last["order_sum"]
    f["last_n_items"] = last["n_items"]
    f["last_disc"] = last["disc"]
    f["last_loy"] = last["loy"]
    f["first_loy"] = first["loy"]
    f["first_order_sum"] = first["order_sum"]
    f["loy_change"] = f["last_loy"] - f["first_loy"]
    f["last_sum_to_mean"] = f["last_order_sum"] / f["order_sum_mean"]

    # --- окна по времени
    for w in WINDOWS:
        m = ot["d"] <= w
        f[f"n_orders_{w}"] = m.groupby(cid).sum()
        f[f"sum_{w}"] = (ot["order_sum"] * m).groupby(cid).sum()
        f[f"n_items_{w}"] = (ot["n_items"] * m).groupby(cid).sum()
    for w in [90, 365]:
        m = ot["d"] <= w
        f[f"loy_max_{w}"] = ot["loy"].where(m).groupby(cid).max()
        f[f"disc_mean_{w}"] = ot["disc"].where(m).groupby(cid).mean()
        f[f"aov_{w}"] = f[f"sum_{w}"] / f[f"n_orders_{w}"].replace(0, np.nan)
    f["n_orders_older_365"] = f["n_orders"] - f["n_orders_365"]

    # --- уровень дня: ритм заказов
    days = ot.groupby(["customer_id", "d"], as_index=False).agg(day_orders=("order_id", "size"), day_sum=("order_sum", "sum"))
    days["gap"] = days.groupby("customer_id")["d"].diff()
    days["rk"] = days.groupby("customer_id").cumcount()
    gd = days.groupby("customer_id")
    f["n_days"] = gd.size()
    f["max_day_orders"] = gd["day_orders"].max()
    f["multi_order_days"] = (days["day_orders"] > 1).groupby(days["customer_id"]).sum()
    f = f.join(gd["gap"].agg(gap_mean="mean", gap_median="median", gap_std="std", gap_min="min", gap_max="max"))
    for k in [1, 2, 4, 9]:
        f[f"d_day_{k + 1}"] = days.loc[days["rk"] == k].set_index("customer_id")["d"]  # давность (k+1)-го с конца дня с заказом
    f["gap_prev"] = days.loc[days["rk"] == 2].set_index("customer_id")["gap"]
    for name, period, limit in [("months_12", 30, 12), ("months_24", 30, 25), ("weeks_13", 7, 13), ("weeks_26", 7, 26)]:
        b = days.assign(b=days["d"] // period)
        f[f"active_{name}"] = b.loc[b["b"] < limit].drop_duplicates(["customer_id", "b"]).groupby("customer_id").size()
    f[[c for c in f.columns if c.startswith("active_")]] = f[[c for c in f.columns if c.startswith("active_")]].fillna(0)

    # --- попадание следующего заказа в целевое окно
    # Время считаем в днях после отсечения: окно — это (GAP_DAYS, GAP_DAYS + win_len].
    r = f["recency"]
    w0, w1 = GAP_DAYS, GAP_DAYS + win_len.reindex(f.index)
    dc = days["customer_id"]
    g, dr = days["gap"], dc.map(r)
    lo, hi = dr + GAP_DAYS, dr + GAP_DAYS + dc.map(win_len)
    n_gaps = g.notna().groupby(dc).sum()
    alive = (g > dr).groupby(dc).sum()  # сколько собственных интервалов клиента длиннее текущей паузы
    f["own_gaps_gt_recency"] = alive
    f["own_share_gaps_gt_recency"] = alive / n_gaps.replace(0, np.nan)
    f["own_p_hit"] = ((g > lo) & (g <= hi)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_hidden"] = ((g > dr) & (g <= lo)).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_p_late"] = (g > hi).groupby(dc).sum() / alive.replace(0, np.nan)
    f["own_share_gaps_le_window"] = (g <= dc.map(win_len)).groupby(dc).sum() / n_gaps.replace(0, np.nan)

    wgt = 0.7 ** (days["rk"] - 1).clip(lower=0)
    f["gap_ewm"] = (g * wgt).groupby(dc).sum() / wgt.where(g.notna()).groupby(dc).sum()
    f["gap_last3"] = g.where(days["rk"].between(1, 3)).groupby(dc).mean()
    f["gap_last"] = days.loc[days["rk"] == 1].set_index("customer_id")["gap"]
    f["gap_last_to_median"] = f["gap_last"] / f["gap_median"]
    for name in ["gap_last", "gap_last3", "gap_ewm", "gap_median", "gap_mean", "gap_min", "gap_max"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r  # ожидаемый день следующего заказа; отрицательный — клиент «опаздывает»
        f[f"tnext_{short}"] = t
        f[f"tnext_{short}_dist"] = (w0 - t).clip(lower=0) + (t - w1).clip(lower=0)  # расстояние до окна, 0 — внутри
        if name in ["gap_last", "gap_ewm", "gap_median", "gap_mean"]:
            f[f"cycles_{short}"] = np.floor((r + w1) / gg) - np.floor((r + w0) / gg)  # сколько раз период укладывается в окно
            f[f"tmod_{short}"] = gg - (r % gg)  # до ближайшего кратного периоду дня, если клиент пропускал циклы

    # --- окно в единицах собственного периода и сглаженное попадание
    for name in ["gap_mean", "gap_median", "gap_last", "gap_last3"]:
        short = name.replace("gap_", "")
        gg = f[name].where(f[name] > 0)
        t = gg - r
        f[f"ulo_{short}"] = (r + w0) / gg  # следующий заказ попадает в окно, если ulo < 1 <= uhi
        f[f"uhi_{short}"] = (r + w1) / gg
        for sd in (10, 20):  # день следующего заказа известен с точностью ± sd дней
            f[f"phit{sd}_{short}"] = ndtr((w1 - t) / sd) - ndtr((w0 - t) / sd)
        f[f"phitrel_{short}"] = ndtr((w1 - t) / (0.25 * gg)) - ndtr((w0 - t) / (0.25 * gg))

    # --- дни с заказами по 30-дневным интервалам и в том же окне год и два назад
    mb = pd.crosstab(dc, (days["d"] // 30).clip(upper=24)).reindex(columns=range(24), fill_value=0)
    mb.columns = [f"mb_{i:02d}" for i in mb.columns]
    f = f.join(mb)
    dl = dc.map(win_len)
    for years, tag in [(1, "ya"), (2, "ya2")]:
        start, end = 365 * years - GAP_DAYS - dl, 365 * years - GAP_DAYS - 1
        f[f"{tag}_days"] = ((days["d"] >= start) & (days["d"] <= end)).groupby(dc).sum()
        f[f"{tag}_days_wide"] = ((days["d"] >= start - 30) & (days["d"] <= end + 30)).groupby(dc).sum()

    # --- производные
    f["formula"] = f["n_orders"] / (f["recency"] + 30)
    f["days_formula"] = f["n_days"] / (f["recency"] + 30)
    f["orders_per_30d"] = f["n_orders"] / (f["tenure"] + 30) * 30
    f["recency_to_gap_mean"] = f["recency"] / f["gap_mean"]
    f["recency_to_gap_median"] = f["recency"] / f["gap_median"]
    f["recency_minus_gap_max"] = f["recency"] - f["gap_max"]
    f["gap_cv"] = f["gap_std"] / f["gap_mean"]
    f["recency_to_tenure"] = f["recency"] / (f["tenure"] + 1)
    f["share_orders_30"] = f["n_orders_30"] / f["n_orders"]
    f["share_orders_90"] = f["n_orders_90"] / f["n_orders"]
    f["share_orders_365"] = f["n_orders_365"] / f["n_orders"]
    f["trend_90_vs_365"] = f["n_orders_90"] / (f["n_orders_365"] - f["n_orders_90"] + 1)
    f["trend_30_vs_90"] = f["n_orders_30"] / (f["n_orders_90"] - f["n_orders_30"] + 1)
    f["share_sum_90"] = f["sum_90"] / f["sum_paid"].replace(0, np.nan)

    # --- уровень позиции: цены, скидки, лояльность, состав покупок
    gl = lines.groupby("customer_id")
    f = f.join(
        gl.agg(
            price_mean=("paid_price", "mean"),
            price_median=("paid_price", "median"),
            price_max=("paid_price", "max"),
            price_min=("paid_price", "min"),
            disc_mean=("discount_percent", "mean"),
            disc_max=("discount_percent", "max"),
            disc_share=("has_disc", "mean"),
            loy_mean=("loyalty_percent", "mean"),
            loy_share=("has_loy", "mean"),
            n_loy_values=("loyalty_percent", "nunique"),
            n_products=("product_id", "nunique"),
            n_brands=("brand", "nunique"),
            n_cat=("category", "nunique"),
            n_cat2=("category2", "nunique"),
            n_gender=("gender_product", "nunique"),
        )
    )
    f["repeat_product_share"] = 1 - f["n_products"] / f["n_items"]
    f["brands_per_item"] = f["n_brands"] / f["n_items"]
    f["cat2_per_item"] = f["n_cat2"] / f["n_items"]

    for col, prefix in [("category", "cat"), ("gender_product", "gender")]:
        share = pd.crosstab(lines["customer_id"], lines[col], normalize="index")
        share.columns = [f"{prefix}_share_{clean(c)}" for c in share.columns]
        f = f.join(share)
        recent = lines.loc[lines["d"] <= 90]
        share = pd.crosstab(recent["customer_id"], recent[col], normalize="index")
        share.columns = [f"{prefix}_share90_{clean(c)}" for c in share.columns]
        f = f.join(share)

    for col in ["category2", "brand"]:
        top = (
            lines.groupby(["customer_id", col], observed=True).size().rename("n").reset_index()
            .sort_values(["customer_id", "n", col], ascending=[True, False, True])
            .drop_duplicates("customer_id").set_index("customer_id")
        )
        f[f"top_{col}_share"] = top["n"] / f["n_items"]
        f[f"top_{col}"] = top[col].astype(str)

    # --- ритм отдельно по категориям, подкатегориям и брендам
    for col in ["category", "category2", "brand"]:
        st = (
            lines[["customer_id", col, "d"]].drop_duplicates()
            .groupby(["customer_id", col], observed=True)["d"].agg(n="size", last="min", first="max")
        )
        st = st[st["n"] >= 2].reset_index()
        st["gap"] = (st["first"] - st["last"]) / (st["n"] - 1)
        st["t"] = st["gap"] - st["last"]
        hi = GAP_DAYS + st["customer_id"].map(win_len)
        st["dist"] = (GAP_DAYS - st["t"]).clip(lower=0) + (st["t"] - hi).clip(lower=0)
        st["hit"] = (st["dist"] == 0).astype(float)
        gs = st.groupby("customer_id")
        f[f"ser_{col}_n"] = gs.size()
        f[f"ser_{col}_hit"] = gs["hit"].sum()
        f[[f"ser_{col}_n", f"ser_{col}_hit"]] = f[[f"ser_{col}_n", f"ser_{col}_hit"]].fillna(0)
        f[f"ser_{col}_hit_share"] = gs["hit"].mean()
        f[f"ser_{col}_min_dist"] = gs["dist"].min()
        best = st.sort_values(["customer_id", "dist", "last"]).drop_duplicates("customer_id").set_index("customer_id")
        f[f"ser_{col}_best_t"] = best["t"]
        f[f"ser_{col}_best_gap"] = best["gap"]

    return f.replace([np.inf, -np.inf], np.nan)


CAT_COLS = ["top_category2", "top_brand"]          # top_brand получает только CatBoost: у него встроенное кодирование средним таргетом
FORCE_KEEP = ["n_days", "recency", "top_category2"]


def cb_frame(df):
    out = df.copy()
    for c in CAT_COLS:   # CatBoost не принимает пропуски в категориальных признаках
        out[c] = out[c].astype(object).where(out[c].notna(), "none").astype(str)
    return out


def set_frames(feats):
    # делает набор признаков текущим для всех стадий
    global tr, te, FR, FR_TE
    data = cust.merge(feats, left_on="customer_id", right_index=True, how="left", validate="1:1")
    assert data["recency"].notna().all() and len(data) == len(cust)
    data["top_category2"] = data["top_category2"].astype("category")
    tr = data[data["part"].eq("train")].reset_index(drop=True)
    te = data[data["part"].eq("test")].set_index("customer_id").loc[sample["customer_id"]].reset_index()
    FR = {"lgb": tr, "xgb": tr, "cb": cb_frame(tr), "lgb_seg": tr}
    FR_TE = {"lgb": te, "xgb": te, "cb": cb_frame(te), "lgb_seg": te}


win_real = cust.set_index("customer_id")["win_len"]
feats_real = build_features(lines, win_real)
set_frames(feats_real)
ALL_FEATURES = [c for c in feats_real.columns if c != "top_brand"]
y = tr["target"].astype(int)
Y, SL = y.to_numpy(), tr["slice"].to_numpy()
print("признаков:", len(ALL_FEATURES), "| строк train:", len(tr), "| строк test:", len(te))
log("признаки", "построены", признаков=len(ALL_FEATURES))

## Схема проверки

In [ ]:
slices_sorted = sorted(tr["slice"].unique())
dev_idx = np.flatnonzero(np.isin(SL, slices_sorted[:-1]))    # январь и февраль: все сравнения и подбор
hold_idx = np.flatnonzero(SL == slices_sorted[-1])           # март: только контроль
all_idx = np.arange(len(tr))
strata = tr["slice"] + "_" + y.astype(str)


def make_folds(n_splits, n_repeats, seed):
    out = []
    for r in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed + r)
        out.append([(dev_idx[a], dev_idx[b]) for a, b in skf.split(dev_idx, strata.iloc[dev_idx])])
    return out


FOLDS = make_folds(N_FOLDS, N_REPEATS, SEED)
TUNE_SPLIT = make_folds(TUNE_FOLDS, 1, SEED + 1000)[0]


def score(rows, pred):
    # среднее AP@20% по срезам, которые есть среди rows; pred идёт в том же порядке, что rows
    pred = np.asarray(pred)
    return float(np.mean([fast_ap(Y[rows][SL[rows] == s], pred[SL[rows] == s]) for s in np.unique(SL[rows])]))


def cols(kind, features):
    return list(features) + (["top_brand"] if kind == "cb" else [])


LGB_FIXED = dict(objective="binary", learning_rate=0.03, n_estimators=5000, subsample_freq=1, cat_smooth=50, n_jobs=-1, verbose=-1)
XGB_FIXED = dict(learning_rate=0.03, n_estimators=5000, tree_method="hist", device="cuda" if GPU else "cpu",
                 enable_categorical=True, eval_metric="logloss", n_jobs=-1, verbosity=0)
CB_FIXED = dict(loss_function="Logloss", learning_rate=0.05, iterations=5000, bootstrap_type="Bayesian",
                task_type="GPU" if GPU else "CPU", thread_count=-1, verbose=0)
DEFAULT = {
    "lgb": dict(num_leaves=31, min_child_samples=100, colsample_bytree=0.5, subsample=0.8, reg_lambda=5.0, reg_alpha=1e-3,
                min_split_gain=0.0, path_smooth=0.0, max_bin=255),
    "xgb": dict(max_depth=6, min_child_weight=20.0, subsample=0.8, colsample_bytree=0.5, reg_lambda=5.0, reg_alpha=1e-3, gamma=0.0, max_bin=256),
    "cb": dict(depth=6, l2_leaf_reg=5.0, random_strength=1.0, bagging_temperature=1.0, border_count=128),
}
SPACE = {
    "lgb": lambda t: dict(
        num_leaves=t.suggest_int("num_leaves", 8, 96, log=True), min_child_samples=t.suggest_int("min_child_samples", 20, 400, log=True),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.2, 0.9), subsample=t.suggest_float("subsample", 0.5, 1.0),
        reg_lambda=t.suggest_float("reg_lambda", 0.1, 50, log=True), reg_alpha=t.suggest_float("reg_alpha", 1e-3, 10, log=True),
        min_split_gain=t.suggest_float("min_split_gain", 0.0, 0.5), path_smooth=t.suggest_float("path_smooth", 0.0, 50.0),
        max_bin=t.suggest_categorical("max_bin", [63, 127, 255])),
    "xgb": lambda t: dict(
        max_depth=t.suggest_int("max_depth", 3, 9), min_child_weight=t.suggest_float("min_child_weight", 1, 200, log=True),
        subsample=t.suggest_float("subsample", 0.5, 1.0), colsample_bytree=t.suggest_float("colsample_bytree", 0.2, 0.9),
        reg_lambda=t.suggest_float("reg_lambda", 0.1, 50, log=True), reg_alpha=t.suggest_float("reg_alpha", 1e-3, 10, log=True),
        gamma=t.suggest_float("gamma", 0.0, 2.0), max_bin=t.suggest_categorical("max_bin", [64, 128, 256])),
    "cb": lambda t: dict(
        depth=t.suggest_int("depth", 4, 9), l2_leaf_reg=t.suggest_float("l2_leaf_reg", 1, 30, log=True),
        random_strength=t.suggest_float("random_strength", 0.1, 5, log=True), bagging_temperature=t.suggest_float("bagging_temperature", 0.0, 2.0),
        border_count=t.suggest_categorical("border_count", [64, 128, 254])),
}


def _fit_one(kind, params, X, yv, X_es, y_es, seed, n_iter):
    # n_iter задан — учим ровно столько деревьев; иначе ранняя остановка по (X_es, y_es)
    if kind == "lgb":
        p = {**LGB_FIXED, **params, "random_state": seed}
        if n_iter:
            m = lgb.LGBMClassifier(**{**p, "n_estimators": int(n_iter)}).fit(X, yv)
            return (lambda Z: m.predict_proba(Z)[:, 1]), int(n_iter)
        m = lgb.LGBMClassifier(**p).fit(X, yv, eval_set=[(X_es, y_es)], eval_metric="binary_logloss", callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
        return (lambda Z: m.predict_proba(Z)[:, 1]), int(m.best_iteration_ or p["n_estimators"])
    if kind == "xgb":
        p = {**XGB_FIXED, **params, "random_state": seed}
        if n_iter:
            m = xgb.XGBClassifier(**{**p, "n_estimators": int(n_iter)}).fit(X, yv, verbose=False)
            return (lambda Z: m.predict_proba(Z)[:, 1]), int(n_iter)
        m = xgb.XGBClassifier(**p, early_stopping_rounds=EARLY_STOP).fit(X, yv, eval_set=[(X_es, y_es)], verbose=False)
        return (lambda Z: m.predict_proba(Z)[:, 1]), int(m.best_iteration) + 1
    p = {**CB_FIXED, **params, "random_seed": seed}
    cats = [c for c in CAT_COLS if c in X.columns]
    if n_iter:
        m = CatBoostClassifier(**{**p, "iterations": int(n_iter)}, cat_features=cats).fit(X, yv)
        return (lambda Z: m.predict_proba(Z)[:, 1]), int(n_iter)
    m = CatBoostClassifier(**p, cat_features=cats).fit(X, yv, eval_set=(X_es, y_es), early_stopping_rounds=EARLY_STOP)
    best = m.get_best_iteration()
    return (lambda Z: m.predict_proba(Z)[:, 1]), int(best + 1 if best is not None else p["iterations"])


SEGMENTS = [lambda d: d["n_days"] <= 3, lambda d: d["n_days"] > 3]


def fit_model(kind, params, X, yv, seed, n_iter=None):
    if kind == "lgb_seg":   # отдельные модели для клиентов с 1–3 и 4+ днями заказов
        parts = []
        for i, seg in enumerate(SEGMENTS):
            m = seg(X).to_numpy()
            parts.append(fit_model("lgb", params, X[m], yv[m], seed, None if n_iter is None else n_iter[i]))

        def predict(Z):
            out = np.zeros(len(Z))
            for seg, (pf, _) in zip(SEGMENTS, parts):
                m = seg(Z).to_numpy()
                if m.any():
                    out[m] = pf(Z[m])
            return out

        return predict, tuple(it for _, it in parts)
    if len(yv) < 200 or yv.nunique() < 2:
        const = float(yv.mean())
        return (lambda Z: np.full(len(Z), const)), 10
    if n_iter:
        return _fit_one(kind, params, X, yv, None, None, seed, max(int(n_iter), 10))
    a, b = train_test_split(np.arange(len(yv)), test_size=0.15, random_state=seed, stratify=yv)
    return _fit_one(kind, params, X.iloc[a], yv.iloc[a], X.iloc[b], yv.iloc[b], seed, None)


def evaluate(kind, params, features, folds=None, hold_seeds=None):
    # кросс-валидация на январе и феврале (значение на каждый фолд каждого повтора) и контроль на марте
    folds = FOLDS if folds is None else folds
    hold_seeds = N_HOLD_SEEDS if hold_seeds is None else hold_seeds
    c, frame = cols(kind, features), FR[kind]
    oof, vals, iters = np.zeros(len(tr)), [], []
    for r, rep in enumerate(folds):
        for a, b in rep:
            pf, it = fit_model(kind, params, frame[c].iloc[a], y.iloc[a], SEED + r)
            p = pf(frame[c].iloc[b])
            oof[b] += p / len(folds)
            vals.append(score(b, p))
            iters.append(it)
    hold = np.zeros(len(hold_idx))
    for s in range(hold_seeds):
        pf, it = fit_model(kind, params, frame[c].iloc[dev_idx], y.iloc[dev_idx], SEED + 100 + s)
        hold += pf(frame[c].iloc[hold_idx]) / hold_seeds
        iters.append(it)
    return dict(kind=kind, params=params, features=list(features), oof=oof, vals=np.array(vals), dev=float(np.mean(vals)),
                hold_pred=hold, hold=score(hold_idx, hold) if hold_seeds else np.nan, iters=np.mean(np.array(iters, dtype=float), axis=0))


def better(new, old):
    # принимаем изменение, если оно не хуже и на кросс-валидации, и на марте, и хотя бы где-то лучше
    d = new["vals"] - old["vals"]
    ok = d.mean() >= -ACCEPT_TOL and new["hold"] >= old["hold"] - ACCEPT_TOL and (d.mean() > ACCEPT_TOL or new["hold"] > old["hold"] + ACCEPT_TOL)
    return ok, f"кросс-валидация {d.mean():+.4f} ± {d.std():.4f} (плюс в {int((d > 0).sum())} из {len(d)}), март {new['hold'] - old['hold']:+.4f}"


base = evaluate("lgb", DEFAULT["lgb"], ALL_FEATURES)
print(f"Исходная точка, LightGBM на всех признаках: кросс-валидация {base['dev']:.4f} ± {base['vals'].std():.4f}, март {base['hold']:.4f}")
log("проверка", "LightGBM, все признаки, исходные параметры", кросс_валидация=base["dev"], март=base["hold"])

# Вариант с одной длиной окна для всех срезов: признаки перестают кодировать номер среза.
# Берём его, если он не хуже и на кросс-валидации, и на марте.
if RUN_WINDOW_CHECK:
    feats_fixed = build_features(lines, pd.Series(WIN_FIXED, index=win_real.index))
    changed = [c for c in ALL_FEATURES if c != "top_category2" and not np.allclose(feats_real[c].to_numpy(float), feats_fixed[c].to_numpy(float), equal_nan=True)]
    print(f"признаков, зависящих от длины окна: {len(changed)}")
    set_frames(feats_fixed)
    cand = evaluate("lgb", DEFAULT["lgb"], ALL_FEATURES)
    d = cand["vals"] - base["vals"]
    take = d.mean() >= -ACCEPT_TOL and cand["hold"] >= base["hold"] - ACCEPT_TOL
    print(f"Фиксированное окно {WIN_FIXED} против настоящего: кросс-валидация {d.mean():+.4f} ± {d.std():.4f} (плюс в {int((d > 0).sum())} из {len(d)}), "
          f"март {cand['hold'] - base['hold']:+.4f} -> {'берём фиксированное' if take else 'оставляем настоящее'}")
    log("окно", f"фиксированная длина окна {WIN_FIXED}", кросс_валидация=cand["dev"], март=cand["hold"], решение="принято" if take else "отклонено")
    if take:
        base = cand
    else:
        set_frames(feats_real)

## Отбор признаков

In [ ]:
FEATURES = list(ALL_FEATURES)
if RUN_SELECTION:
    Xd, yd = tr[ALL_FEATURES].iloc[dev_idx], Y[dev_idx]
    sel_params = dict(objective="binary", learning_rate=0.05, n_estimators=200, num_leaves=31, min_child_samples=100,
                      subsample=0.8, subsample_freq=1, colsample_bytree=0.5, n_jobs=-1, verbose=-1)
    gain = lambda target, seed: lgb.LGBMClassifier(**sel_params, random_state=seed).fit(Xd, target).booster_.feature_importance("gain")
    rng = np.random.default_rng(SEED)
    real = np.mean([gain(yd, s) for s in range(3)], axis=0)
    null = np.array([gain(rng.permutation(yd), s) for s in range(4 if QUICK else 10)])
    ratio = pd.Series(real / (np.percentile(null, 90, axis=0) + 1e-9), index=ALL_FEATURES)
    keep = [f for f in ALL_FEATURES if ratio[f] > 1.0 or f in FORCE_KEEP]
    print(f"Важность выше шума у {len(keep)} признаков из {len(ALL_FEATURES)}")
    print("Самые слабые:", ratio.sort_values().head(15).round(2).to_dict())
    if len(keep) < len(ALL_FEATURES):
        cand = evaluate("lgb", DEFAULT["lgb"], keep)
        ok, text = better(cand, base)
        d = cand["vals"] - base["vals"]
        # меньше признаков при том же качестве тоже годится: быстрее подбор и меньше шума
        take = ok or (d.mean() >= -ACCEPT_TOL and cand["hold"] >= base["hold"] - ACCEPT_TOL)
        print("Отобранные против всех:", text, "->", "берём отобранные" if take else "оставляем все")
        log("отбор", f"{len(keep)} признаков из {len(ALL_FEATURES)}", кросс_валидация=cand["dev"], март=cand["hold"], решение="принято" if take else "отклонено")
        if take:
            FEATURES, base = keep, cand
print("признаков в работе:", len(FEATURES))

## Подбор параметров

In [ ]:
def tune(kind):
    c, frame = cols(kind, FEATURES), FR[kind]

    def objective(trial):
        params = SPACE[kind](trial)
        vals = []
        for i, (a, b) in enumerate(TUNE_SPLIT):
            pf, _ = fit_model(kind, params, frame[c].iloc[a], y.iloc[a], SEED)
            vals.append(score(b, pf(frame[c].iloc[b])))
            trial.report(float(np.mean(vals)), i)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(vals))

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=0))
    study.enqueue_trial(DEFAULT[kind])   # исходные параметры — первая попытка
    study.optimize(objective, n_trials=N_TRIALS[kind], timeout=TUNE_TIMEOUT)
    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    print(f"{kind}: попыток {len(study.trials)}, досчитано {len(done)}, лучшая {study.best_value:.4f}, исходные параметры {done[0].value:.4f}")
    return sorted(done, key=lambda t: -t.value)


RES = {}
KINDS = ["lgb", "xgb", "cb"]
for kind in KINDS:
    try:
        best = base if kind == "lgb" else evaluate(kind, DEFAULT[kind], FEATURES)
        log("модель", f"{kind}, исходные параметры", кросс_валидация=best["dev"], март=best["hold"])
        print(f"\n{kind}, исходные параметры: кросс-валидация {best['dev']:.4f}, март {best['hold']:.4f}")
        if RUN_TUNING:
            t0 = time.time()
            trials = tune(kind)
            seen = [DEFAULT[kind]]
            for t in trials:
                if len(seen) > N_TOP:
                    break
                if t.params in seen:
                    continue
                seen.append(t.params)
                cand = evaluate(kind, t.params, FEATURES)
                ok, text = better(cand, best)
                print(f"  попытка {t.number} ({t.value:.4f} в подборе): {text} -> {'принята' if ok else 'отклонена'}")
                log("подбор", f"{kind}, попытка {t.number}", кросс_валидация=cand["dev"], март=cand["hold"], решение="принято" if ok else "отклонено")
                if ok:
                    best = cand
            print(f"  подбор и перепроверка: {(time.time() - t0) / 60:.1f} мин; параметры: {best['params']}")
        RES[kind] = best
    except Exception as e:   # сбой одной библиотеки не должен останавливать весь прогон
        print(f"!!! {kind} пропущен: {type(e).__name__}: {e}")
        log("модель", f"{kind} пропущен", решение=f"ошибка: {type(e).__name__}")

if "lgb" in RES:
    RES["lgb_seg"] = evaluate("lgb_seg", RES["lgb"]["params"], FEATURES)
    log("модель", "lgb_seg, параметры lgb", кросс_валидация=RES["lgb_seg"]["dev"], март=RES["lgb_seg"]["hold"])

display(pd.DataFrame({k: {"кросс-валидация": r["dev"], "разброс по фолдам": r["vals"].std(), "март": r["hold"], "деревьев": str(np.round(r["iters"]).astype(int).tolist())} for k, r in RES.items()}).T)

## Смесь моделей

In [ ]:
def rank_in_slice(rows, pred):
    return pd.Series(pred).groupby(SL[rows]).rank(pct=True).to_numpy()


names = list(RES)
dev_mat = np.column_stack([rank_in_slice(dev_idx, RES[k]["oof"][dev_idx]) for k in names])
hold_mat = np.column_stack([rank_in_slice(hold_idx, RES[k]["hold_pred"]) for k in names])
equal = np.full(len(names), 1 / len(names))
grid = [np.array(w) / 10 for w in itertools.product(range(11), repeat=len(names)) if sum(w) == 10]
w_best = max(grid, key=lambda w: score(dev_idx, dev_mat @ w))
blend_rows = {
    **{k: {"янв + фев, вне фолда": score(dev_idx, dev_mat[:, i]), "март": score(hold_idx, hold_mat[:, i])} for i, k in enumerate(names)},
    "равная смесь": {"янв + фев, вне фолда": score(dev_idx, dev_mat @ equal), "март": score(hold_idx, hold_mat @ equal)},
    "подобранные веса": {"янв + фев, вне фолда": score(dev_idx, dev_mat @ w_best), "март": score(hold_idx, hold_mat @ w_best)},
}
display(pd.DataFrame(blend_rows).T)
use_tuned_w = blend_rows["подобранные веса"]["март"] >= blend_rows["равная смесь"]["март"] - ACCEPT_TOL
WEIGHTS = w_best if use_tuned_w else equal
print("подобранные веса:", dict(zip(names, np.round(w_best, 2).tolist())), "->", "берём их" if use_tuned_w else "берём равную смесь")
print("Корреляция прогнозов моделей на марте (Спирмен):")
display(pd.DataFrame(hold_mat, columns=names).corr(method="spearman"))
for k in ["равная смесь", "подобранные веса"]:
    log("смесь", k, кросс_валидация=blend_rows[k]["янв + фев, вне фолда"], март=blend_rows[k]["март"])

res_h = pd.DataFrame({"target": Y[hold_idx], "n_days": tr["n_days"].to_numpy()[hold_idx]})
res_h["pct"] = pd.Series(hold_mat @ WEIGHTS).rank(pct=True, ascending=False).to_numpy()
print("Март, итоговая смесь: доля target = 1 по положению в рейтинге")
display(res_h.groupby(pd.cut(res_h["pct"], [0, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 1.0]), observed=True)["target"].agg(["mean", "size"]).T)

## Финальное обучение и отправки

In [ ]:
test_rank = {}
for k in names:
    r = RES[k]
    c = cols(k, r["features"])
    scale = 1.25   # в финале данных больше, чем было у внутренней обучающей части, поэтому деревьев чуть больше
    n_iter = tuple(int(round(v * scale)) for v in np.atleast_1d(r["iters"])) if k == "lgb_seg" else int(round(float(r["iters"]) * scale))
    pred = np.zeros(len(te))
    for s in range(N_SEEDS):
        pf, _ = fit_model(k, r["params"], FR[k][c], y, SEED + 500 + s, n_iter=n_iter)
        pred += pf(FR_TE[k][c]) / N_SEEDS
    test_rank[k] = pd.Series(pred).rank(pct=True).to_numpy()
    print(k, "деревьев:", n_iter, "| зёрен:", N_SEEDS)

test_mat = np.column_stack([test_rank[k] for k in names])


def save(scores, fname):
    sub = pd.DataFrame({"customer_id": te["customer_id"].to_numpy(), "score": pd.Series(scores).rank(pct=True).to_numpy()})
    assert sub["customer_id"].tolist() == sample["customer_id"].tolist() and sub["score"].between(0, 1).all()
    sub.to_csv(OUT_DIR / fname, index=False)


save(test_mat @ WEIGHTS, "submission.csv")            # основная: смесь, выбранная по марту
save(test_mat @ equal, "submission_equal.csv")        # осторожная: равная смесь
for i, k in enumerate(names):
    save(test_mat[:, i], f"submission_{k}.csv")

pd.DataFrame({"customer_id": tr["customer_id"], "slice": SL, "target": Y, **{k: RES[k]["oof"] for k in names}}).to_parquet(OUT_DIR / "oof_dev.parquet", index=False)
pd.DataFrame({"customer_id": te["customer_id"], **test_rank}).to_parquet(OUT_DIR / "test_pred.parquet", index=False)
json.dump({"features": FEATURES, "weights": dict(zip(names, map(float, WEIGHTS))),
           "models": {k: {"params": RES[k]["params"], "iters": np.atleast_1d(RES[k]["iters"]).tolist()} for k in names}},
          open(OUT_DIR / "pipeline_config.json", "w"), ensure_ascii=False, indent=1, default=float)
log("финал", "отправки сохранены")
print("Корреляция прогнозов на тесте (Спирмен):")
display(pd.DataFrame(test_mat, columns=names).corr(method="spearman"))
print(f"Всего времени: {(time.time() - T0) / 60:.1f} мин")
journal = pd.DataFrame(LOG)
journal.to_csv(OUT_DIR / "journal.csv", index=False)
display(journal)